In [1]:
import numpy as np
import tensorflow as tf
import pandas as pd

2025-02-27 10:14:19.252760: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-02-27 10:14:19.273399: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1740631459.297480 4031430 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1740631459.304568 4031430 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-27 10:14:19.328830: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
from tqdm import tqdm
from transformers import BertTokenizerFast, AutoTokenizer
from datasets import load_dataset
import glob

In [4]:
tamil_file = "Tamil_Corpus_Cleaned.txt"

with open(tamil_file, encoding='utf-8') as f:  
    print(f.read(5000))

ஒலிம்பிக் போட்டிகள் நடந்த இடங்கள் 1. 1896 - ஏதென்ஸ், கிரீஸ் 2. 1900 - பாரிஸ், பிரான்ஸ் 3. 1904 - செயின் லூயிஸ், அமெரிக்கா 4. 1908 - லண்டன்,பிரிட்டன் 5. 1912 - ஸ்டோக்ஹோம், சுவீடன் 6. 1920 - ஆண்ட்வெர்ப், பெல்ஜியம் 7. 1924 - பாரிஸ், பிரான்ஸ் 8. 1928 - ஆம்ஸ்டர்டாம், ஹாலந்து 9. 1932 - லாஸ், ஏஞ்சல்ஸ் 10. 1936 - பெர்லின், ஜெர்மனி 11. 1948 - லண்டன், இங்கிலாந்து 12. 1952 - ஹல்சின்கி, பின்லாந்து 13. 1956 - மேபோர்ன்,ஆஸ்திரேலியா 14. 1960 - ரோம், இத்தாலி 15. 1964 - டோக்கியோ, ஜப்பான் 16. 1968 - மெக்சிகோ, மெக்ஸிக்கோ 17. 1972 - மியூனிக், ஜெர்மனி 18. 1976 - மான்ட்ரியல், கனடா 19. 1980 - மாஸ்கோ, USSR 20. 1984 - லாஸ் ஏஞ்சல்ஸ், அமெரிக்கா 21. 1988 - சியோல், தென் கொரியா 22. 1992 - பார்சிலோனா, ஸ்பெயின் 23. 1996 - அட்லாண்டா, அமெரிக்கா 24. 2000 - சிட்னி, ஆஸ்திரேலியா 25. 2004 - ஏதென்ஸ், கிரீஸ் 26. 2008 - பீஜிங், சீனா 27. 2012 - லண்டன், இங்கிலாந்து 28. 2016 - ரியோ, பிரேசில் 29. 2020 - டோக்கியோ, ஜப்பான் 30. 2024 - பாரிஸ், பிரான்ஸ் 31. 2028 - லாஸ் ஏஞ்சல்ஸ், அமெரிக்கா Music ncs:
இன்னும் வாழ்வதில் நம்பிக்கையற்றுப்போ

In [5]:
import os

file_path = "Tamil_Corpus_Cleaned.txt"
print("File exists:", os.path.exists(file_path))
print("File size:", os.path.getsize(file_path), "bytes")

File exists: True
File size: 5454682875 bytes


In [6]:
from datasets import load_dataset

dataset = load_dataset('text', data_files="Tamil_Corpus_Cleaned.txt", encoding="utf-8")

In [7]:
dataset['train']

Dataset({
    features: ['text'],
    num_rows: 6971580
})

In [14]:
from tqdm import tqdm
from transformers import BertTokenizerFast, AutoTokenizer
from datasets import load_dataset
import glob

#loading bert tokenizer to work as a base for the new tokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

def batch_iterator(batch_size=10000):
    for i in tqdm(range(0, len(dataset['train']), batch_size)):
        yield dataset['train'][i: i +batch_size]['text']
bert_tokenizer = tokenizer.train_new_from_iterator(text_iterator=batch_iterator(), 
                                                   vocab_size=20000)
bert_tokenizer.save_pretrained('bert_wordpiece/')

100%|█████████████████████████████████████████████████████████████████████████████████| 698/698 [03:36<00:00,  3.23it/s]


('bert_wordpiece/tokenizer_config.json',
 'bert_wordpiece/special_tokens_map.json',
 'bert_wordpiece/vocab.txt',
 'bert_wordpiece/added_tokens.json',
 'bert_wordpiece/tokenizer.json')

In [ ]:
# cell-3
#tokenizing the whole text  

from datasets import load_dataset
import glob
import tokenizers
from transformers import Trainer, TrainingArguments, LineByLineTextDataset, BertModel
from transformers import BertConfig, BertForMaskedLM, DataCollatorForLanguageModeling
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('bert_wordpiece/')
max_seq_length = 64

dataset = load_dataset('text', data_files="Tamil_Corpus_Cleaned.txt", encoding="utf-8")

def encode_with_truncation(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length",
                   max_length=max_seq_length, return_special_tokens_mask=True)


dataset = dataset["train"].map(encode_with_truncation, batched=True)
dataset.set_format(type="torch", columns=["input_ids", "attention_mask"])

display(len(dataset))

In [ ]:
dataset.features

In [17]:
dataset = dataset.remove_columns(["text"])

In [18]:
dataset.save_to_disk("tokenized_dataset_wordpiece/dataset")

Saving the dataset (0/7 shards):   0%|          | 0/6971580 [00:00<?, ? examples/s]

In [19]:
#start here
from datasets import load_from_disk 
from datasets import load_dataset
import glob
import tokenizers
from transformers import Trainer, TrainingArguments, LineByLineTextDataset, BertModel
from transformers import BertConfig, BertForMaskedLM, DataCollatorForLanguageModeling
from transformers import AutoTokenizer

dataset = load_from_disk("tokenized_dataset_wordpiece/dataset")

In [20]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained('bert_wordpiece/')

# Model configuration optimized for efficiency
config = BertConfig(
    vocab_size=20000,  
    hidden_size=192,  
    num_hidden_layers=3,  
    num_attention_heads=3,  
    max_position_embeddings=64
)

# Initialize model
model = BertForMaskedLM(config)
print(f"Total Parameters: {model.num_parameters()}")


Total Parameters: 7906208


In [24]:
import glob
import tokenizers

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=True, mlm_probability=0.15)
epochs = 1
save_steps = 10000 #save checkpoint every 10000 steps
batch_size = 64

training_args = TrainingArguments(
    output_dir = 'bert_wordpiece/',
    overwrite_output_dir=True,
    num_train_epochs = epochs,
    per_device_train_batch_size = batch_size,
    save_steps = save_steps,
    save_total_limit = 2, #only save the last 5 checkpoints
    fp16=True,
    learning_rate = 1e-4,  # 5e-5 is the default
    logging_steps = 5_000,
    # gradient_accumulation_steps=4,
    # gradient_checkpointing=True
)
trainer = Trainer(
    model = model,
    args = training_args,
    data_collator=data_collator,
    train_dataset=dataset
)
# Train model
trainer.train()
# Save final model
trainer.save_model('bert_wordpiece/')

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Step,Training Loss
5000,7.702900
10000,6.878700
15000,6.235100


In [4]:
tamil_file = "cleaned_tamil_test.txt"

with open(tamil_file, encoding='utf-8') as f:  
    print(f.read(500))

கொண்டலாம்பட்டி
கொண்டலாம்பட்டி (ஆங்கிலம்:Kondalampatti), என்பது இந்தியாவின் தமிழ்நாடு மாநிலத்தில் அமைந்துள்ள சேலம் மாவட்டத்தில் இருக்கும் ஒரு கணக்கெடுப்பில் உள்ள ஊர் ஆகும்.
இந்திய 2001 மக்கள் தொகை கணக்கெடுப்பின்படி 16,300 மக்கள் இங்கு வசிக்கின்றார்கள். இவர்களில் 50% ஆண்கள், 50% பெண்கள் ஆவார்கள். கொண்டல்பட்டி மக்களின் சராசரி கல்வியறிவு 52% ஆகும், இதில் ஆண்களின் கல்வியறிவு 61%, பெண்களின் கல்வியறிவு 42% ஆகும். இது இந்திய தேசிய சராசரி கல்வியறிவான 59.5% விட குறைந்ததே. கொண்டல்பட்டி மக்கள் தொகையில் 13% 


In [6]:
from transformers import BertTokenizer

def tokenize_text(file_path, model='bert_wordpiece/'):
    tokenizer = AutoTokenizer.from_pretrained(model)
    
    with open(file_path, 'r', encoding='utf-8') as file:
        text = file.read()
    
    tokens = tokenizer.tokenize(text)
    
    return tokens

tokenized_text = tokenize_text('cleaned_tamil_test.txt')

Token indices sequence length is longer than the specified maximum sequence length for this model (1612655 > 512). Running this sequence through the model will result in indexing errors


## Vocabualry 50000

In [8]:
from tqdm import tqdm
from transformers import BertTokenizerFast, AutoTokenizer
from datasets import load_dataset
import glob

#loading bert tokenizer to work as a base for the new tokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

def batch_iterator(batch_size=10000):
    for i in tqdm(range(0, len(dataset['train']), batch_size)):
        yield dataset['train'][i: i +batch_size]['text']
bert_tokenizer = tokenizer.train_new_from_iterator(text_iterator=batch_iterator(), 
                                                   vocab_size=50000)
bert_tokenizer.save_pretrained('bert_wordpiece_50000/')

100%|█████████████████████████████████████████████████████████████████████████████████| 698/698 [02:48<00:00,  4.13it/s]


('bert_wordpiece_50000/tokenizer_config.json',
 'bert_wordpiece_50000/special_tokens_map.json',
 'bert_wordpiece_50000/vocab.txt',
 'bert_wordpiece_50000/added_tokens.json',
 'bert_wordpiece_50000/tokenizer.json')

In [11]:
# cell-3
#tokenizing the whole text  

from datasets import load_dataset
import glob
import tokenizers
from transformers import Trainer, TrainingArguments, LineByLineTextDataset, BertModel
from transformers import BertConfig, BertForMaskedLM, DataCollatorForLanguageModeling
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('bert_wordpiece_50000/')
max_seq_length = 64

dataset = load_dataset('text', data_files="Tamil_Corpus_Cleaned.txt", encoding="utf-8")

def encode_with_truncation(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length",
                   max_length=max_seq_length, return_special_tokens_mask=True)


dataset = dataset["train"].map(encode_with_truncation, batched=True)
dataset.set_format(type="torch", columns=["input_ids", "attention_mask"])

display(len(dataset))

Map:   0%|          | 0/6971580 [00:00<?, ? examples/s]

6971580

In [12]:
dataset.features

{'text': Value(dtype='string', id=None),
 'input_ids': Sequence(feature=Value(dtype='int32', id=None), length=-1, id=None),
 'token_type_ids': Sequence(feature=Value(dtype='int8', id=None), length=-1, id=None),
 'attention_mask': Sequence(feature=Value(dtype='int8', id=None), length=-1, id=None),
 'special_tokens_mask': Sequence(feature=Value(dtype='int8', id=None), length=-1, id=None)}

In [13]:
dataset = dataset.remove_columns(["text"])

In [14]:
dataset.save_to_disk("tokenized_dataset_wordpiece_50000/dataset")

Saving the dataset (0/7 shards):   0%|          | 0/6971580 [00:00<?, ? examples/s]

In [17]:
#start here

from datasets import load_from_disk 

from datasets import load_dataset
import glob
import tokenizers
from transformers import Trainer, TrainingArguments, LineByLineTextDataset, BertModel
from transformers import BertConfig, BertForMaskedLM, DataCollatorForLanguageModeling
from transformers import AutoTokenizer

dataset = load_from_disk("tokenized_dataset_wordpiece_50000/dataset")

In [18]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained('bert_wordpiece_50000/')

# Model configuration optimized for efficiency
config = BertConfig(
    vocab_size=50000,  
    hidden_size=192,  
    num_hidden_layers=3,  
    num_attention_heads=3,  
    max_position_embeddings=64
)

# Initialize model
model = BertForMaskedLM(config)
print(f"Total Parameters: {model.num_parameters()}")


Total Parameters: 13696208


In [19]:
import glob
import tokenizers

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=True,
                                               mlm_probability=0.15)
epochs = 1
save_steps = 10000 #save checkpoint every 10000 steps
batch_size = 64

training_args = TrainingArguments(
    output_dir = 'bert_wordpiece_50000/',
    overwrite_output_dir=True,
    num_train_epochs = epochs,
    per_device_train_batch_size = batch_size,
    save_steps = save_steps,
    save_total_limit = 2, #only save the last 5 checkpoints
    fp16=True,
    learning_rate = 1e-4,  # 5e-5 is the default
    logging_steps = 5_000,
    # gradient_accumulation_steps=4,
    # gradient_checkpointing=True
)

trainer = Trainer(
    model = model,
    args = training_args,
    data_collator=data_collator,
    train_dataset=dataset

)

# Train model
trainer.train()

# Save final model
trainer.save_model('bert_wordpiece_50000/')

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Step,Training Loss
5000,8.214200
10000,7.340700
15000,6.783900
